# PHASE 3: Model Training with Hybrid Features (48 Features)

 
 Purpose: Train and evaluate 4 ML models with hybrid feature set
 
Models:
1. LightGBM 
2. Random Forest 
3. XGBoost 
4. Logistic Regression 
# 
Features: 48 total
- 36 Oh et al. features (forensic patterns, temporal, cross-artifact)
- 9 timestamp parsing features (before/after/direction from old RF)
- 3 source classification features (one-hot encoding from old RF)

Goal: See if hybrid approach improves precision while maintaining recall
 
Baseline (36 features, LightGBM):
- F1: 0.6368
- Recall: 95.9%
- Precision: 47.65%


## Setup and Configuration

**Dataset**: Phase 2 output (all_cases_combined_features.csv)
- 88,190 files from 18 training datasets (12 PE + 6 APT training)
- 266 suspicious files (0.3%)
- 87,924 benign files (99.7%)
- 48 engineered features

**New Features Added** (Phase 2 v2.0):
1. event_count - Total forensic events per filename
2. events_in_1min_window - Events within ±1 minute
3. events_in_5min_window - Events within ±5 minutes  
4. matches_system_pattern - WindowsUpdate, OneDrive, Dropbox patterns
5. is_system_file_type - System file extensions (.etl, .tmp, .sdb, etc.)

**Train/Test Split Strategy**: Grouped Stratified (Case-Based)
- Keep entire datasets together (no file-level splitting within datasets)
- Training: ~14 datasets
- Testing: ~4 datasets
- Rationale: Tests generalization to NEW attack scenarios (production deployment scenario)

**Algorithms**:
1. Random Forest - Ensemble baseline
2. XGBoost - Gradient boosting
3. LightGBM - Fast gradient boosting
4. Logistic Regression - Linear baseline

**Evaluation Metrics** (appropriate for extreme imbalance):
1. F1-Score - Harmonic mean of precision and recall
2. Recall (Detection Rate) - Of all attacks, how many caught?
3. Precision - Of flagged files, how many are real attacks?
4. False Negative Rate (FNR) - What % of attacks missed? (security risk)
5. False Positive Rate (FPR) - What % of benign files flagged? (analyst workload)

**Note**: Accuracy and AUC-ROC omitted due to extreme imbalance (naive "all benign" model achieves 99.7% accuracy)


In [279]:
# Cell 1: Import Libraries
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print(f"LightGBM version: {lgb.__version__}")
print(f"XGBoost version: {xgb.__version__}")


Libraries imported successfully
LightGBM version: 4.6.0
XGBoost version: 3.1.2


In [280]:
# Cell 2: Define Paths
BASE_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis")

# Input: Phase 2 features
PHASE2_OUTPUT = BASE_DIR / "data/processed/Phase 2 - Features/all_cases_combined_features.csv"

# Output directory for Phase 3
PHASE3_DIR = BASE_DIR / "data/processed/Phase 3 - Model Training"
PHASE3_DIR.mkdir(parents=True, exist_ok=True)

# Model output directory
MODEL_DIR = BASE_DIR / "models/for autopsy"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input file: {PHASE2_OUTPUT}")
print(f"Output directory: {PHASE3_DIR}")
print(f"Model directory: {MODEL_DIR}")


Input file: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 2 - Features/all_cases_combined_features.csv
Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training
Model directory: /Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy


## Load and Prepare Features

**Key steps:**
1. Load 48 features from Phase 2
2. Convert timestamp strings to Unix timestamps (numeric)
3. Split into train/test (80/20)
4. Handle class imbalance


In [281]:
# Cell 3: Load Phase 2 Features
print("Loading Phase 2 features...")
df = pd.read_csv(PHASE2_OUTPUT)

print(f"Loaded {len(df):,} records")
print(f"Total columns: {len(df.columns)}")
print(f"Suspicious events: {df['ground_truth_label'].sum():,}")
print(f"Benign events: {(df['ground_truth_label'] == 0).sum():,}")
print(f"Class imbalance: {df['ground_truth_label'].sum() / len(df) * 100:.2f}% suspicious")


Loading Phase 2 features...
Loaded 88,190 records
Total columns: 62
Suspicious events: 266
Benign events: 87,924
Class imbalance: 0.30% suspicious


In [282]:
# Cell 4: Define Feature List (48 Features)
feature_cols_list = [
    # Category 1: Forensic Patterns (16 features - Oh et al.)
    'zero_in_nanoseconds_lf',
    'zero_in_nanoseconds_suspicious',
    'zero_in_nanoseconds',
    'time_reversal_event',
    'basic_info_changed',
    'using_another_timestamp',
    'si_timestamp_changed',
    'update_resident_value',
    'creation_time_modified',
    'modified_time_modified',
    'accessed_time_modified',
    'mft_time_modified',
    'timestamp_changed_to_past',
    'multiple_timestamps_changed',
    'same_as_another_file',
    'zero_nano_time_reversal',
    
    # Category 2: Cross-Artifact Validation (4 features - Oh et al.)
    'cross_artifact_detected',
    'has_logfile_evidence',
    'has_usnjrnl_evidence',
    'cross_artifact_validation_score',
    
    # Category 3: Temporal Features (3 features - Oh et al.)
    'event_count',
    'events_in_1min_window',
    'events_in_5min_window',
    
    # Category 4: File Characteristics (9 features - Oh et al.)
    'is_executable',
    'is_document',
    'is_archive',
    'is_image',
    'path_depth',
    'filename_length',
    'in_temp_directory',
    'in_system_directory',
    'in_program_files',
    
    # Category 5: Timestamp Features (2 features - Oh et al.)
    'has_timestamp_data',
    'timestamp_source',
    
    # Category 6: System Pattern Recognition (2 features - Data-Driven)
    'matches_system_pattern',
    'is_system_file_type',
    
    # Category 7: Timestamp Parsing (9 features - old Random Forest)
    'lf_creation_time_before',
    'lf_creation_time_after',
    'creation_time_changed_to_past',
    'lf_modified_time_before',
    'lf_modified_time_after',
    'modified_time_changed_to_past',
    'lf_accessed_time_before',
    'lf_accessed_time_after',
    'accessed_time_changed_to_past',
    
    # Category 8: Source Classification (3 features - old Random Forest)
    'source_logfile_only',
    'source_usnjrnl_only',
    'source_both',
]

print(f"Total features: {len(feature_cols_list)}")
assert len(feature_cols_list) == 48, f"ERROR: Expected 48 features, got {len(feature_cols_list)}"


Total features: 48


In [283]:
# Cell 5: Prepare Features and Target
print("\nPreparing features and target...")

# Extract features and target
X = df[feature_cols_list].copy()
y = df['ground_truth_label'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:")
print(y.value_counts())

# Verify all features exist
missing_features = [f for f in feature_cols_list if f not in X.columns]
if missing_features:
    print(f"\nERROR: Missing features: {missing_features}")
    raise ValueError(f"Missing features in dataset")
else:
    print(f"\nAll 48 features present in dataset")



Preparing features and target...
Feature matrix shape: (88190, 48)
Target distribution:
ground_truth_label
0    87924
1      266
Name: count, dtype: int64

All 48 features present in dataset


In [284]:
# Cell 6: Convert Timestamp Strings to Numeric
print("\nConverting timestamp strings to Unix timestamps...")

timestamp_cols = [
    'lf_creation_time_before', 'lf_creation_time_after',
    'lf_modified_time_before', 'lf_modified_time_after',
    'lf_accessed_time_before', 'lf_accessed_time_after'
]

for col in timestamp_cols:
    if col in X.columns:
        print(f"  Converting {col}...")
        # Convert to datetime
        X[col] = pd.to_datetime(X[col], errors='coerce')
        # Convert to Unix timestamp (seconds since epoch)
        X[col] = X[col].astype('int64') / 10**9
        # Fill missing with 0
        X[col] = X[col].fillna(0)
        
        non_zero = (X[col] != 0).sum()
        print(f"    Non-zero values: {non_zero:,} ({non_zero/len(X)*100:.2f}%)")

print(f"\nTimestamp conversion complete")
print(f"Feature matrix shape: {X.shape}")
print(f"Data types:")
print(X.dtypes.value_counts())



Converting timestamp strings to Unix timestamps...
  Converting lf_creation_time_before...
    Non-zero values: 88,190 (100.00%)
  Converting lf_creation_time_after...
    Non-zero values: 88,190 (100.00%)
  Converting lf_modified_time_before...
    Non-zero values: 88,190 (100.00%)
  Converting lf_modified_time_after...
    Non-zero values: 88,190 (100.00%)
  Converting lf_accessed_time_before...
    Non-zero values: 88,190 (100.00%)
  Converting lf_accessed_time_after...
    Non-zero values: 88,190 (100.00%)

Timestamp conversion complete
Feature matrix shape: (88190, 48)
Data types:
int64      40
float64     7
bool        1
Name: count, dtype: int64


In [285]:
# Cell 7: Check for Missing Values
print("\nChecking for missing values...")

missing = X.isna().sum()
if missing.sum() > 0:
    print(f"WARNING: Found {missing.sum()} missing values")
    print("\nFeatures with missing values:")
    print(missing[missing > 0])
    
    # Fill missing with 0 (conservative approach)
    print("\nFilling missing values with 0...")
    X = X.fillna(0)
else:
    print("No missing values found")

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Final target shape: {y.shape}")



Checking for missing values...
No missing values found

Final feature matrix shape: (88190, 48)
Final target shape: (88190,)


In [286]:
# Cell 8: Train/Test Split
print("\nSplitting data into train/test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Maintain class balance
)

print(f"Training set: {len(X_train):,} events")
print(f"  Suspicious: {y_train.sum():,} ({y_train.sum()/len(y_train)*100:.2f}%)")
print(f"  Benign: {(y_train == 0).sum():,} ({(y_train == 0).sum()/len(y_train)*100:.2f}%)")

print(f"\nTest set: {len(X_test):,} events")
print(f"  Suspicious: {y_test.sum():,} ({y_test.sum()/len(y_test)*100:.2f}%)")
print(f"  Benign: {(y_test == 0).sum():,} ({(y_test == 0).sum()/len(y_test)*100:.2f}%)")



Splitting data into train/test sets...
Training set: 70,552 events
  Suspicious: 213 (0.30%)
  Benign: 70,339 (99.70%)

Test set: 17,638 events
  Suspicious: 53 (0.30%)
  Benign: 17,585 (99.70%)


## Model Training

**Train 4 models with 48 hybrid features:**

1. **LightGBM**
2. **Random Forest**
3. **XGBoost**
4. **Logistic Regression**

**Evaluation metrics:**
- Precision: Of flagged events, how many are truly suspicious?
- Recall: Of truly suspicious events, how many did we catch?
- F1-Score: Harmonic mean of precision and recall

**Target:**
- Recall: Greater than or equal to 95% (maintain high recall)
- Precision: Greater than 50% (improve from 47.65%)
- F1: Greater than 0.65 (improve from 0.6368)


In [287]:
# Cell 9: Model 1 - LightGBM
print("\n" + "="*80)
print("MODEL 1: LightGBM (Baseline Comparison)")
print("="*80)

print("\nTraining LightGBM...")

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train == 0).sum() / y_train.sum()
print(f"Class imbalance ratio: {scale_pos_weight:.2f}")

lgb_model = lgb.LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=7,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train)

# Predictions
y_pred_lgb = lgb_model.predict(X_test)
y_pred_proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

# Metrics
lgb_precision = precision_score(y_test, y_pred_lgb)
lgb_recall = recall_score(y_test, y_pred_lgb)
lgb_f1 = f1_score(y_test, y_pred_lgb)

print("\n" + "-"*80)
print("LightGBM Results:")
print("-"*80)
print(f"Precision: {lgb_precision:.4f} ({lgb_precision*100:.2f}%)")
print(f"Recall:    {lgb_recall:.4f} ({lgb_recall*100:.2f}%)")
print(f"F1-Score:  {lgb_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lgb, target_names=['Benign', 'Suspicious']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))

print("\nComparison to baseline (36 features):")
print(f"  Previous F1:        0.6368")
print(f"  Current F1:         {lgb_f1:.4f}")
print(f"  Improvement:        {(lgb_f1 - 0.6368)*100:+.2f} percentage points")
print(f"\n  Previous Precision: 47.65%")
print(f"  Current Precision:  {lgb_precision*100:.2f}%")
print(f"  Improvement:        {(lgb_precision*100 - 47.65):+.2f} percentage points")
print(f"\n  Previous Recall:    95.9%")
print(f"  Current Recall:     {lgb_recall*100:.2f}%")
print(f"  Change:             {(lgb_recall*100 - 95.9):+.2f} percentage points")



MODEL 1: LightGBM (Baseline Comparison)

Training LightGBM...
Class imbalance ratio: 330.23



--------------------------------------------------------------------------------
LightGBM Results:
--------------------------------------------------------------------------------
Precision: 0.0063 (0.63%)
Recall:    0.0755 (7.55%)
F1-Score:  0.0117

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.96      0.98     17585
  Suspicious       0.01      0.08      0.01        53

    accuracy                           0.96     17638
   macro avg       0.50      0.52      0.50     17638
weighted avg       0.99      0.96      0.98     17638


Confusion Matrix:
[[16959   626]
 [   49     4]]

Comparison to baseline (36 features):
  Previous F1:        0.6368
  Current F1:         0.0117
  Improvement:        -62.51 percentage points

  Previous Precision: 47.65%
  Current Precision:  0.63%
  Improvement:        -47.02 percentage points

  Previous Recall:    95.9%
  Current Recall:     7.55%
  Change:             -88.35 percentage po

In [288]:
# Cell 10: Model 2 - Random Forest
print("\n" + "="*80)
print("MODEL 2: Random Forest (Old Model Architecture)")
print("="*80)

print("\nTraining Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',  # Handle imbalance
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Metrics
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("\n" + "-"*80)
print("Random Forest Results:")
print("-"*80)
print(f"Precision: {rf_precision:.4f} ({rf_precision*100:.2f}%)")
print(f"Recall:    {rf_recall:.4f} ({rf_recall*100:.2f}%)")
print(f"F1-Score:  {rf_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Benign', 'Suspicious']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))



MODEL 2: Random Forest (Old Model Architecture)

Training Random Forest...

--------------------------------------------------------------------------------
Random Forest Results:
--------------------------------------------------------------------------------
Precision: 0.4815 (48.15%)
Recall:    0.9811 (98.11%)
F1-Score:  0.6460

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     17585
  Suspicious       0.48      0.98      0.65        53

    accuracy                           1.00     17638
   macro avg       0.74      0.99      0.82     17638
weighted avg       1.00      1.00      1.00     17638


Confusion Matrix:
[[17529    56]
 [    1    52]]


In [289]:
# Cell 11: Model 3 - XGBoost
print("\n" + "="*80)
print("MODEL 3: XGBoost (Alternative Gradient Boosting)")
print("="*80)

print("\nTraining XGBoost...")

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=7,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
xgb_precision = precision_score(y_test, y_pred_xgb)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)

print("\n" + "-"*80)
print("XGBoost Results:")
print("-"*80)
print(f"Precision: {xgb_precision:.4f} ({xgb_precision*100:.2f}%)")
print(f"Recall:    {xgb_recall:.4f} ({xgb_recall*100:.2f}%)")
print(f"F1-Score:  {xgb_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Benign', 'Suspicious']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))



MODEL 3: XGBoost (Alternative Gradient Boosting)

Training XGBoost...

--------------------------------------------------------------------------------
XGBoost Results:
--------------------------------------------------------------------------------
Precision: 0.7879 (78.79%)
Recall:    0.9811 (98.11%)
F1-Score:  0.8739

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     17585
  Suspicious       0.79      0.98      0.87        53

    accuracy                           1.00     17638
   macro avg       0.89      0.99      0.94     17638
weighted avg       1.00      1.00      1.00     17638


Confusion Matrix:
[[17571    14]
 [    1    52]]


In [290]:
# Cell 12: Model 4 - Logistic Regression
print("\n" + "="*80)
print("MODEL 4: Logistic Regression (Linear Baseline)")
print("="*80)

print("\nTraining Logistic Regression...")

lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    verbose=0
)

lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Metrics
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

print("\n" + "-"*80)
print("Logistic Regression Results:")
print("-"*80)
print(f"Precision: {lr_precision:.4f} ({lr_precision*100:.2f}%)")
print(f"Recall:    {lr_recall:.4f} ({lr_recall*100:.2f}%)")
print(f"F1-Score:  {lr_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Benign', 'Suspicious']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))



MODEL 4: Logistic Regression (Linear Baseline)

Training Logistic Regression...

--------------------------------------------------------------------------------
Logistic Regression Results:
--------------------------------------------------------------------------------
Precision: 0.0824 (8.24%)
Recall:    0.9434 (94.34%)
F1-Score:  0.1515

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.97      0.98     17585
  Suspicious       0.08      0.94      0.15        53

    accuracy                           0.97     17638
   macro avg       0.54      0.96      0.57     17638
weighted avg       1.00      0.97      0.98     17638


Confusion Matrix:
[[17028   557]
 [    3    50]]


## Model Comparison

**Compare all 4 models across key metrics:**
- Which model has best F1-Score?
- Which model has best precision (reduce false positives)?
- Which model maintains high recall (catch all suspicious events)?


In [291]:
# Cell 13: Model Comparison with Confusion Matrix Analysis
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

# Calculate confusion matrix components for each model
def get_confusion_matrix_metrics(y_true, y_pred):
    """Calculate TP, FP, TN, FN from predictions"""
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'TP': tp,  # True Positives (correctly caught suspicious)
        'FP': fp,  # False Positives (benign flagged as suspicious)
        'TN': tn,  # True Negatives (correctly identified benign)
        'FN': fn   # False Negatives (missed suspicious events)
    }

# Get metrics for all models
lgb_cm = get_confusion_matrix_metrics(y_test, y_pred_lgb)
rf_cm = get_confusion_matrix_metrics(y_test, y_pred_rf)
xgb_cm = get_confusion_matrix_metrics(y_test, y_pred_xgb)
lr_cm = get_confusion_matrix_metrics(y_test, y_pred_lr)

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Model': ['LightGBM', 'Random Forest', 'XGBoost', 'Logistic Regression'],
    'Precision': [lgb_precision, rf_precision, xgb_precision, lr_precision],
    'Recall': [lgb_recall, rf_recall, xgb_recall, lr_recall],
    'F1-Score': [lgb_f1, rf_f1, xgb_f1, lr_f1],
    'TP': [lgb_cm['TP'], rf_cm['TP'], xgb_cm['TP'], lr_cm['TP']],
    'FP': [lgb_cm['FP'], rf_cm['FP'], xgb_cm['FP'], lr_cm['FP']],
    'TN': [lgb_cm['TN'], rf_cm['TN'], xgb_cm['TN'], lr_cm['TN']],
    'FN': [lgb_cm['FN'], rf_cm['FN'], xgb_cm['FN'], lr_cm['FN']]
})

# Format as percentages
comparison_df['Precision (%)'] = comparison_df['Precision'] * 100
comparison_df['Recall (%)'] = comparison_df['Recall'] * 100

# Sort by F1-Score
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("\nPerformance Metrics (sorted by F1-Score):")
print(comparison_df[['Model', 'Precision (%)', 'Recall (%)', 'F1-Score']].to_string(index=False))

print("\n" + "-"*80)
print("CONFUSION MATRIX BREAKDOWN:")
print("-"*80)
print("\nTP = True Positives (correctly caught suspicious events)")
print("FP = False Positives (benign events flagged as suspicious)")
print("TN = True Negatives (correctly identified benign events)")
print("FN = False Negatives (missed suspicious events)")

print("\n")
print(comparison_df[['Model', 'TP', 'FP', 'TN', 'FN']].to_string(index=False))

# Total suspicious events in test set
total_suspicious = y_test.sum()
total_benign = (y_test == 0).sum()

print("\n" + "-"*80)
print("DETAILED BREAKDOWN BY MODEL:")
print("-"*80)

for idx, row in comparison_df.iterrows():
    model_name = row['Model']
    tp, fp, tn, fn = row['TP'], row['FP'], row['TN'], row['FN']
    
    print(f"\n{model_name}:")
    print(f"  Suspicious events (total: {total_suspicious}):")
    print(f"    Caught (TP):  {tp}/{total_suspicious} ({tp/total_suspicious*100:.1f}%)")
    print(f"    Missed (FN):  {fn}/{total_suspicious} ({fn/total_suspicious*100:.1f}%)")
    
    print(f"  Benign events (total: {total_benign:,}):")
    print(f"    Correct (TN): {tn:,}/{total_benign:,} ({tn/total_benign*100:.1f}%)")
    print(f"    False alarm (FP): {fp:,}/{total_benign:,} ({fp/total_benign*100:.1f}%)")

# Identify best model
best_model_name = comparison_df.iloc[0]['Model']
best_f1 = comparison_df.iloc[0]['F1-Score']
best_precision = comparison_df.iloc[0]['Precision (%)']
best_recall = comparison_df.iloc[0]['Recall (%)']
best_tp = comparison_df.iloc[0]['TP']
best_fp = comparison_df.iloc[0]['FP']
best_fn = comparison_df.iloc[0]['FN']

print("\n" + "="*80)
print(f"BEST MODEL: {best_model_name}")
print("="*80)
print(f"F1-Score:  {best_f1:.4f}")
print(f"Precision: {best_precision:.2f}%")
print(f"Recall:    {best_recall:.2f}%")

print(f"\nConfusion Matrix:")
print(f"  Caught suspicious: {best_tp}/{total_suspicious} events ({best_tp/total_suspicious*100:.1f}%)")
print(f"  Missed suspicious: {best_fn}/{total_suspicious} events ({best_fn/total_suspicious*100:.1f}%)")
print(f"  False alarms: {best_fp:,} benign events flagged")

print("\n" + "-"*80)
print("COMPARISON TO BASELINE (36 features, LightGBM):")
print("-"*80)
print(f"Baseline F1:        0.6368")
print(f"Best Model F1:      {best_f1:.4f}")
print(f"Improvement:        {(best_f1 - 0.6368)*100:+.2f} percentage points")
print(f"\nBaseline Precision: 47.65%")
print(f"Best Model Precision: {best_precision:.2f}%")
print(f"Improvement:        {(best_precision - 47.65):+.2f} percentage points")
print(f"\nBaseline Recall:    95.9%")
print(f"Best Model Recall:  {best_recall:.2f}%")
print(f"Change:             {(best_recall - 95.9):+.2f} percentage points")

if best_f1 > 0.6368:
    print("\nRESULT: Hybrid features IMPROVED performance!")
elif best_f1 > 0.63:
    print("\nRESULT: Hybrid features maintained similar performance")
else:
    print("\nRESULT: Hybrid features did not improve performance")

print("\n" + "="*80)



MODEL COMPARISON SUMMARY

Performance Metrics (sorted by F1-Score):
              Model  Precision (%)  Recall (%)  F1-Score
            XGBoost      78.787879   98.113208  0.873950
      Random Forest      48.148148   98.113208  0.645963
Logistic Regression       8.237232   94.339623  0.151515
           LightGBM       0.634921    7.547170  0.011713

--------------------------------------------------------------------------------
CONFUSION MATRIX BREAKDOWN:
--------------------------------------------------------------------------------

TP = True Positives (correctly caught suspicious events)
FP = False Positives (benign events flagged as suspicious)
TN = True Negatives (correctly identified benign events)
FN = False Negatives (missed suspicious events)


              Model  TP  FP    TN  FN
            XGBoost  52  14 17571   1
      Random Forest  52  56 17529   1
Logistic Regression  50 557 17028   3
           LightGBM   4 626 16959  49

----------------------------------------

In [292]:
# Cell 15: Save Best Model
print("\n" + "="*80)
print("SAVING BEST MODEL")
print("="*80)

# Select best model based on F1 scores
model_scores = {
    'LightGBM': lgb_f1,
    'Random Forest': rf_f1,
    'XGBoost': xgb_f1,
    'Logistic Regression': lr_f1
}

best_model_name = max(model_scores, key=model_scores.get)
best_f1 = model_scores[best_model_name]

# Get best model and its metrics
if best_model_name == 'LightGBM':
    best_model = lgb_model
    best_precision = lgb_precision * 100
    best_recall = lgb_recall * 100
    feature_importance = lgb_model.feature_importances_
elif best_model_name == 'Random Forest':
    best_model = rf_model
    best_precision = rf_precision * 100
    best_recall = rf_recall * 100
    feature_importance = rf_model.feature_importances_
elif best_model_name == 'XGBoost':
    best_model = xgb_model
    best_precision = xgb_precision * 100
    best_recall = xgb_recall * 100
    feature_importance = xgb_model.feature_importances_
else:
    best_model = lr_model
    best_precision = lr_precision * 100
    best_recall = lr_recall * 100
    feature_importance = np.abs(lr_model.coef_[0])

# Create importance dataframe
importance_df = pd.DataFrame({
    'Feature': feature_cols_list,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print(f"\nBest model: {best_model_name}")
print(f"F1-Score: {best_f1:.4f}")
print(f"Precision: {best_precision:.2f}%")
print(f"Recall: {best_recall:.2f}%")

# Save model
model_path = MODEL_DIR / "best_model_hybrid_48features.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

print(f"\nBest model saved: {model_path}")

# Save feature columns
feature_cols_path = MODEL_DIR / "feature_columns_hybrid_48features.pkl"
with open(feature_cols_path, 'wb') as f:
    pickle.dump(feature_cols_list, f)

print(f"Feature columns saved: {feature_cols_path}")

# Save feature importance
importance_path = PHASE3_DIR / "feature_importance_hybrid_48features.csv"
importance_df.to_csv(importance_path, index=False)
print(f"Feature importance saved: {importance_path}")

# Save comparison results
comparison_df = pd.DataFrame({
    'Model': ['LightGBM', 'Random Forest', 'XGBoost', 'Logistic Regression'],
    'Precision': [lgb_precision, rf_precision, xgb_precision, lr_precision],
    'Recall': [lgb_recall, rf_recall, xgb_recall, lr_recall],
    'F1-Score': [lgb_f1, rf_f1, xgb_f1, lr_f1]
})
comparison_df['Precision (%)'] = comparison_df['Precision'] * 100
comparison_df['Recall (%)'] = comparison_df['Recall'] * 100
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

comparison_path = PHASE3_DIR / "model_comparison_hybrid_48features.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Model comparison saved: {comparison_path}")

print("\n" + "="*80)
print("MODEL TRAINING COMPLETE")
print("="*80)
print(f"\nBest model: {best_model_name}")
print(f"F1-Score: {best_f1:.4f}")
print(f"Precision: {best_precision:.2f}%")
print(f"Recall: {best_recall:.2f}%")

print("\nComparison to baseline (36 features, LightGBM):")
print(f"  Baseline F1:        0.6368")
print(f"  Current F1:         {best_f1:.4f}")
print(f"  Improvement:        {(best_f1 - 0.6368)*100:+.2f} percentage points")
print(f"\n  Baseline Precision: 47.65%")
print(f"  Current Precision:  {best_precision:.2f}%")
print(f"  Improvement:        {(best_precision - 47.65):+.2f} percentage points")
print(f"\n  Baseline Recall:    95.9%")
print(f"  Current Recall:     {best_recall:.2f}%")
print(f"  Change:             {(best_recall - 95.9):+.2f} percentage points")



SAVING BEST MODEL

Best model: XGBoost
F1-Score: 0.8739
Precision: 78.79%
Recall: 98.11%

Best model saved: /Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy/best_model_hybrid_48features.pkl
Feature columns saved: /Users/soni/Github/Digital-Detectives_Thesis/models/for autopsy/feature_columns_hybrid_48features.pkl
Feature importance saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/feature_importance_hybrid_48features.csv
Model comparison saved: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training/model_comparison_hybrid_48features.csv

MODEL TRAINING COMPLETE

Best model: XGBoost
F1-Score: 0.8739
Precision: 78.79%
Recall: 98.11%

Comparison to baseline (36 features, LightGBM):
  Baseline F1:        0.6368
  Current F1:         0.8739
  Improvement:        +23.71 percentage points

  Baseline Precision: 47.65%
  Current Precision:  78.79%
  Improvement:        +31.14 percentage points

  Baseline R